In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QUICK-PDE: دالة Qiskit من ColibriTD
*راجع [مرجع API](https://docs.quantum.ibm.com/api/functions/colibritd-pde)*

> **Note:** دوال Qiskit ميزة تجريبية متاحة لمستخدمي خطة IBM Quantum&reg; Premium وخطة Flex وخطة On-Prem (عبر واجهة برمجة IBM Quantum Platform). هذه الميزات في مرحلة الإصدار التجريبي وقابلة للتغيير.
## نظرة عامة
حلّال المعادلات التفاضلية الجزئية (PDE) المقدَّم هنا جزء من منصة Quantum Innovative Computing Kit (QUICK) الخاصة بنا (QUICK-PDE)، وهو مغلَّف كدالة Qiskit. بواسطة دالة QUICK-PDE، يمكنك حل المعادلات التفاضلية الجزئية المتخصصة بمجالات محددة على وحدات معالجة الكم (QPUs) من IBM Quantum. تستند هذه الدالة إلى الخوارزمية الموصوفة في [ورقة H-DES من ColibriTD](https://arxiv.org/abs/2410.01130). وتستطيع هذه الخوارزمية حل مسائل الفيزياء المتعددة المعقدة، بدءًا من ديناميكيات الموائع الحسابية (CFD) وتشوه المواد (MD)، مع حالات استخدام أخرى قيد التطوير.

للتعامل مع المعادلات التفاضلية، تُرمَّز الحلول التجريبية على شكل تركيبات خطية من الدوال المتعامدة (عادةً كثيرات حدود شيبيشيف، وتحديدًا $2^n$ منها، حيث $n$ هو عدد الـ qubits التي ترمّز دالتك)، ومُعامَلة بزوايا دائرة كمية متغيرة (VQC). ينتج الـ ansatz حالة ترمّز الدالة، ويُقيَّم عبر مُراقِبات تتيح مجموعاتها تقييم الدالة عند جميع النقاط. يمكنك بعد ذلك تقييم دالة الخسارة التي تُرمَّز فيها المعادلات التفاضلية، وضبط الزوايا في حلقة هجينة كما هو موضح أدناه. تقترب الحلول التجريبية تدريجيًا من الحلول الفعلية حتى تصل إلى نتيجة مُرضية.

![مسار عمل دالة QUICK-PDE](../docs/images/guides/colibritd-equation-solver/diagram.svg)

إضافةً إلى هذه الحلقة الهجينة، يمكنك أيضًا تسلسل محسِّنات مختلفة. يفيد هذا عندما تريد محسِّنًا شاملًا للعثور على مجموعة جيدة من الزوايا، ثم محسِّنًا أدق لاتباع التدرج نحو أفضل مجموعة من الزوايا المجاورة. في حالة ديناميكيات الموائع الحسابية (CFD)، يُعطي تسلسل التحسين الافتراضي أفضل النتائج - أما في حالة تشوه المواد (MD)، فبينما يُعطي الافتراضي نتائج جيدة، يمكنك ضبطه أكثر للحصول على فوائد خاصة بالمسألة.

لاحظ أنه لكل متغير من متغيرات الدالة، نحدد عدد الـ qubits (يمكنك تجربتها). بتكديس 10 دوائر متطابقة وتقييم 10 مراقِبات متطابقة على qubits مختلفة عبر دائرة كبيرة واحدة، يمكنك تخفيف الضوضاء ضمن عملية تحسين CMA معتمدًا على طريقة تعلم الضوضاء، وتقليل عدد القياسات المطلوبة بشكل ملحوظ.
### ديناميكيات الموائع الحسابية
معادلة Burgers' اللاعكوسة تُنمذج الموائع غير اللزجة المتدفقة كما يلي:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0,$$

$u$ يُمثّل حقل سرعة الموائع. لهذه الحالة شرط حدودي زمني: يمكنك اختيار الشرط الابتدائي ثم السماح للنظام بالاسترخاء. حاليًا، الشروط الابتدائية المقبولة فقط هي الدوال الخطية: $ax + b$.

وسيطات المعادلات التفاضلية لـ CFD تقع على شبكة ثابتة كما يلي:

- $t$ بين 0 و 0.95 بـ 30 نقطة عينة. $x$ بين 0 و 0.95 بخطوة حجمها 0.2375.

### تشوه المواد
تركّز هذه الحالة على التشوه المرن الهاز أحادي البعد (اختبار الشد)، حيث يُثبَّت قضيب في الفضاء ويُسحب من طرفه الآخر. نصف المسألة كما يلي:

$$u' - \frac{\sigma}{3K} - \frac{2}{\sqrt{3}}\epsilon_0\left(\frac{\sigma'}{\sigma_0\sqrt{3}}\right)^n = 0$$

$$\sigma' - b = 0,$$

$K$ يمثّل معامل الانتفاخ للمادة المُشدَّدة، $n$ الأُس في قانون القوى، $b$ القوة لكل وحدة كتلة، $\epsilon_0$ حد الإجهاد التناسبي، $\sigma_0$ حد الانفعال التناسبي، $u$ دالة الإجهاد، و$\sigma$ دالة الانفعال.

القضيب المدروس له طول وحدوي. لهذه الحالة شرط حدودي لإجهاد السطح $t$، أي مقدار الشغل اللازم لشد القضيب.

وسيطات المعادلات التفاضلية لـ MD تقع على شبكة ثابتة كما يلي:

- $x$ بين 0 و 1 بخطوة حجمها 0.04.
## المعايير القياسية
يعرض الجدول التالي إحصائيات حول تشغيلات مختلفة لدالتنا.

| المثال                      | عدد الـ qubits | التهيئة        | الخطأ     | الوقت الإجمالي (دقيقة) | استخدام وقت التشغيل (دقيقة) |
| ---------------------------- | ---------------- | --------------------- | --------- | ---------------- | ------------------- |
| معادلة Burgers' اللاعكوسة   | 50               | `PHYSICALLY_INFORMED` | $10^{-2}$ | 66               | 25                  |
| اختبار الشد أحادي البعد المرن الهاز  | 18               | `RANDOM`              | $10^{-2}$ | 123              | 100                 |
## ابدأ الآن
أكمل [النموذج لطلب الوصول إلى دالة QUICK-PDE](https://forms.cloud.microsoft/e/3Wi9cbjQPK). ثم، بافتراض أنك [حفظت حسابك](/guides/functions#install-qiskit-functions-catalog-client) في بيئتك المحلية، اختر الدالة كما يلي:

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

In [ ]:
quick = catalog.load("colibritd/quick-pde")

تحقق من [حالة](/guides/functions#check-job-status) عبء عمل دالة Qiskit أو استرجع [النتائج](/guides/functions#retrieve-results) كما يلي:

In [ ]:
# launch the simulation with initial conditions u(0,x) = a*x + b
job = quick.run(use_case="cfd", physical_parameters={"a": 1.0, "b": 0.0})

Check your Qiskit Function workload's [status](/docs/guides/functions-get-started#check-job-status) or return [results](/docs/guides/functions-get-started#retrieve-results) as follows:

In [ ]:
# Print the ID so you can use it later, if necessary
print(job.job_id)
print(job.status())
solution = job.result()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

_ = plt.figure()
ax = plt.axes(projection="3d")

# plot the solution using the 3d plotting capabilities of pyplot
t, x = np.meshgrid(solution["samples"]["t"], solution["samples"]["x"])
ax.plot_surface(
    t,
    x,
    solution["functions"]["u"],
    edgecolor="royalblue",
    lw=0.25,
    rstride=26,
    cstride=26,
    alpha=0.3,
)
ax.scatter(t, x, solution["functions"]["u"], marker=".")
ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

plt.show()

![ناتج خلية الكود السابقة](../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif)

### تشوه المواد
تتطلب حالة تشوه المواد المعاملات الفيزيائية للمادة والقوة المطبّقة، كما يلي:

In [ ]:
import matplotlib.pyplot as plt

# select the properties of your material
job = quick.run(
    use_case="md",
    physical_parameters={
        "t": 12.0,
        "K": 100.0,
        "n": 4.0,
        "b": 10.0,
        "epsilon_0": 0.1,
        "sigma_0": 5.0,
    },
)

# plot the result
solution = job.result()

_ = plt.figure()
stress_plot = plt.subplot(211)
plt.plot(solution["samples"]["x"], solution["functions"]["u"])
strain_plot = plt.subplot(212)
plt.plot(solution["samples"]["x"], solution["functions"]["sigma"])

plt.show()

![ناتج خلية الكود السابقة](../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif)

المثال التالي يوضح كيفية الحصول على قيمة الدالة عند مجموعة محددة من الإحداثيات:

In [ ]:
# u(t=0.2, x=0.7) == 2
assert solution["samples"]["t"][1] == 0.2
assert solution["samples"]["x"][2] == 0.7
assert solution["functions"]["u"][1, 2] == 2

## استرجاع رسائل الخطأ
إذا كانت حالة عبء عملك `ERROR`، استخدم `job.error_message()` لاسترجاع رسالة الخطأ للمساعدة في تصحيح الأخطاء، كما يلي:

In [ ]:
job = quick.run(use_case="mdf", physical_params={})

print(job.error_message())


# or write a wrapper around it for a more human readable version
def pprint_error(job):
    print("".join(eval(job.error_message())["error"]))


print("___")
pprint_error(job)

## Get support

For support, contact qiskit-function-support@colibritd.com.

## Next steps

<Admonition type="tip" title="Recommendations">
- Fill out the form to [request access to the QUICK-PDE function](https://forms.cloud.microsoft/e/3Wi9cbjQPK).
- Visit the [API reference](/docs/api/functions/colibritd-pde) for this Qiskit Function.
- Try modeling a flowing non-viscous fluid using QUICK-PDE in the [tutorial](/docs/tutorials/colibritd-pde).
- Review [Jaffali, H., et al. (2025).  H-DES: a Quantum-Classical Hybrid Differential Equation Solver. arXiv preprint arXiv:2410.01130](https://arxiv.org/abs/2410.01130).
</Admonition>